### This notebook compute "P3. Erosivity" indicator for the 27 basins of IKI Project

**Created:** Sophia Bakar (sbakar@rti.org)  

**N/A Handling:** No missing data. Where monthly or annual precipitation may be 0 mm, the calculation is skipped in the for loop to avoid dividing by zero.    
 
**Notes:**  
General Methodology: 
1. Pull total monthly and annual precipitation for each COMID.
2. Use precipitation totals to calculate total erosivity per year and COMID based on total precipitation. Calculations use the Arnoldus method to estimate rainfall erosivity (R).  
3. Take the average across all years per COMID to calculate mean erosivity for the given scenario.  
 

In [1]:
import numpy as np
import pandas as pd
import sqlite3
import geopandas as gpd
import yaml
from pathlib import Path

In [ ]:
# set master path for input data from config file

config_path = Path("../../config.yaml")

with open(config_path, "r") as f:
    config = yaml.safe_load(f)

master_path = Path(config["master_path"])

In [ ]:
# set paths for input data and indicators database 
subbasins_shapefile = master_path / "Modelacion" / "Grupos_Modelacion" / "GIS_WaterALLOC_General" / "Peru_AHD_with_districts.shp"
db_path = master_path / "Indicadores" / "BD_RiesgoClimatico_IKI.db"

In [ ]:
IndID = 103 #Indicator ID (Peligro = 1 + 0X where X is the Peligro Indicator number) 

scenarios = {
    1: {"met_source_id": 2},  # Baseline
    2: {"met_source_id": 5},  # Future
}

subbasins_gdf = gpd.read_file(subbasins_shapefile).set_index('COMID').to_crs('WGS84')

In [5]:
# conversion for cm to mm
cm_to_mm = 10.0

In [ ]:
precip_records = []   # store all COMID-year-month results

for grupo, relative_path in config["modeling_groups"].items():
        
    sqlite_path = master_path / relative_path
    conn = sqlite3.connect(sqlite_path)

    for ScnID, scn_info in scenarios.items():

        query = f"""
        SELECT
            comid,
            avg_precip_cm,
            measured_date
        FROM catchment_met_observations
        WHERE met_source_id = {scn_info['met_source_id']};
        """

        met = pd.read_sql_query(query, conn)

        if met.empty:
            continue

        met['measured_date'] = pd.to_datetime(
            met['measured_date'],
            format='%Y-%m-%d %H:%M:%S %z UTC',
            errors='coerce'
        )
        met = met.dropna(subset=['measured_date'])

        met['precip_mm'] = met['avg_precip_cm'] * cm_to_mm
        met['year'] = met['measured_date'].dt.year
        met['month'] = met['measured_date'].dt.month
        met['ScnID'] = ScnID

        # Monthly totals
        monthly = (
            met.groupby(['ScnID', 'comid', 'year', 'month'])['precip_mm']
            .sum()
            .reset_index()
            .rename(columns={'precip_mm': 'Pi_mm'})
        )

        # Annual totals
        annual = (
            met.groupby(['ScnID', 'comid', 'year'])['precip_mm']
            .sum()
            .reset_index()
            .rename(columns={'precip_mm': 'P_annual_mm'})
        )

        merged = monthly.merge(
            annual,
            on=['ScnID', 'comid', 'year'],
            how='left'
        )

        precip_records.append(merged)

    conn.close()

# %%
df_precip = pd.concat(precip_records, ignore_index=True)

In [ ]:
def compute_erosivity_R(monthly_df):
    """
    Input: monthly_df = 12-row dataframe for a single COMID-year
           with columns ['Pi_mm', 'P_annual_mm']
    Output: R value for that COMID-year
    """
    R_sum = 0
    
    p = monthly_df['P_annual_mm'].iloc[0]

    for _, row in monthly_df.iterrows():
        Pi = row['Pi_mm']

        # avoid divide-by-zero
        if p == 0 or Pi == 0:
            continue  

        term = 1.735 * 10 ** (1.5 * np.log10((Pi**2) / p) - 0.08188)
        R_sum += term

    return R_sum

In [ ]:
# Compute R for each COMID-year
R_per_year = (
    df_precip
    .groupby(['ScnID', 'comid', 'year'])
    .apply(compute_erosivity_R)
    .reset_index(name='R_annual')
)


# Compute mean annual R per COMID
mean_R_per_comid = (
    R_per_year
    .groupby(['ScnID', 'comid'])['R_annual']
    .mean()
    .reset_index(name='R_mean')
)

In [ ]:
conn = sqlite3.connect(db_path)
cursor = conn.cursor()

rows_to_insert = []

for _, row in mean_R_per_comid.iterrows():
    rows_to_insert.append((
        int(row['ScnID']),
        IndID,
        int(row['comid']),
        float(row['R_mean'])
    ))

insert_query = """
INSERT OR REPLACE INTO IndValues_Dyn (ScnID, IndID, COMID, Value)
VALUES (?, ?, ?, ?);
"""


In [ ]:
# check that min and max values match the expected range based on the Indicators Table 
indicator_limits = pd.read_sql_query(
    """
    SELECT IndID, Min, Max
    FROM Indicators
    WHERE IndID = ?
    """,
    conn,
    params=(IndID,)
)

if indicator_limits.empty:
    raise ValueError(f"No entry found in Indicators table for IndID = {IndID}")

ind_min = indicator_limits.loc[0, 'Min']
ind_max = indicator_limits.loc[0, 'Max']

print(f"Indicator {IndID}  Min: {ind_min}, Max: {ind_max}")

value_stats = (
    mean_R_per_comid['R_mean']
    .agg(['min', 'max', 'count'])
    .reset_index()
)

print("\n=== Values to be inserted (by scenario) ===")
print(value_stats)

# Check for duplicates in the input dataframe before insert
df_check = pd.DataFrame(rows_to_insert, columns=['ScnID', 'IndID', 'COMID', 'Value'])
duplicates = df_check.duplicated(subset=['ScnID', 'IndID', 'COMID'])
print("Duplicates in rows_to_insert:", df_check[duplicates])

In [ ]:
#Execute insert to SQLite Database
cursor.executemany(insert_query, rows_to_insert)
conn.commit()
conn.close()